# Aquaponics + Vermiponics Simulation – Interactive Notebook

Enhanced closed-loop model with temperature scaling, dissolved oxygen, vermicomposting, alkalinity dynamics, and approximate pH effects.

## Setup

In [ ]:
%matplotlib inline
import numpy as np
from scipy.integrate import odeint
import matplotlib.pyplot as plt
from ipywidgets import interact

## Model Definition

In [ ]:
def aquaponics_vermiponics_model(y, t, params):
    F, P, A, I, N, S, DO, Alk = y
    
    (V, T, Q10, r_feed_base, e_feed_to_solid, e_solid_to_ammonia,
     k1_max, Km1, k2_max, Km2, r_p_max, Km_p,
     r_f_base, K_f, r_p, K_p, m_f_base,
     k_worm_base, k_reaer, k_DO_fish, k_DO_decay,
     DO_crit, DO_half, alk_cons_amox, alk_add_uptake) = params
    
    # Temperature correction
    Q = Q10 ** ((T - 20.0) / 10.0)
    
    # Approximate pH from alkalinity
    pH = 6.2 + (Alk / 100.0) * 0.6
    pH_fact = max(0.0, min(1.0, (pH - 5.5) / 2.0))
    
    # Fish growth and mortality
    r_f = r_f_base * Q
    m_f = m_f_base * (1 + (DO_crit - DO) / DO_half if DO < DO_crit else 0)
    dF_dt = r_f * F * (1 - F / K_f) - m_f * F
    
    # Feed to solids
    feed = r_feed_base * F * Q
    dS_dt = e_feed_to_solid * feed * 1000 / V
    
    # Vermicomposting
    worm_activity = k_worm_base * Q * (DO / (DO_half + DO))
    dS_dt -= worm_activity * S
    dA_dt = worm_activity * S * e_solid_to_ammonia
    
    # Nitrification (limited by DO, temp, pH)
    nitr1 = k1_max * Q * (DO / (DO_half + DO)) * A / (Km1 + A) * pH_fact
    nitr2 = k2_max * Q * (DO / (DO_half + DO)) * I / (Km2 + I) * pH_fact
    dA_dt -= nitr1
    dI_dt = nitr1 - nitr2
    dN_dt = nitr2
    
    # Plant uptake
    uptake = r_p_max * P * Q * N / (Km_p + N)
    dP_dt = r_p * (uptake / 1000) * (1 - P / K_p)
    dN_dt -= uptake / V
    
    # Alkalinity change
    dAlk_dt = -alk_cons_amox * nitr1 + alk_add_uptake * (uptake / V)
    
    # Dissolved oxygen
    reaer = k_reaer * (8.5 - DO)
    fish_cons = k_DO_fish * F * Q / V
    bact_cons = 3.43 * nitr1 + 1.14 * nitr2
    decay_cons = k_DO_decay * worm_activity * S
    dDO_dt = reaer - fish_cons - bact_cons - decay_cons
    
    return [dF_dt, dP_dt, dA_dt, dI_dt, dN_dt, dS_dt, dDO_dt, dAlk_dt]

## Base Parameters & Initial Conditions

In [ ]:
params_base = [
    200.0,   # V (L)
    22.0,    # T (°C)
    2.5,     # Q10
    0.025,   # r_feed_base
    0.60,    # e_feed_to_solid
    0.35,    # e_solid_to_ammonia
    3.0,     # k1_max (ammonia oxidation)
    1.0,     # Km1
    6.0,     # k2_max (nitrite oxidation)
    0.7,     # Km2
    0.8,     # r_p_max (nitrate uptake)
    12.0,    # Km_p
    0.015,   # r_f_base
    12.0,    # K_f (fish carrying capacity kg)
    0.008,   # r_p (plant growth scaling)
    25.0,    # K_p (plant cap kg)
    0.0015,  # m_f_base
    0.15,    # k_worm_base
    0.8,     # k_reaer
    0.12,    # k_DO_fish
    0.0003,  # k_DO_decay
    4.0,     # DO_crit
    2.0,     # DO_half
    7.14,    # alk_cons_amox
    3.57     # alk_add_uptake
]

y0_base = [1.2, 0.8, 0.1, 0.0, 8.0, 5.0, 7.5, 150.0]  # F, P, A, I, N, S, DO, Alk

## Interactive Simulation

In [ ]:
@interact(T=(15, 30, 0.5), initial_Alk=(50, 300, 10))
def run_sim(T=22.0, initial_Alk=150.0):
    params = params_base.copy()
    params[1] = T                  # update temperature (index 1)
    
    y0 = y0_base.copy()
    y0[7] = initial_Alk            # update initial alkalinity (index 7)
    
    t = np.linspace(0, 120, 1200)
    solution = odeint(aquaponics_vermiponics_model, y0, t, args=(params,))
    
    F, P, A, I, N, S, DO, Alk = solution.T
    pH_approx = 6.2 + (Alk / 100.0) * 0.6
    
    fig, axs = plt.subplots(5, 1, figsize=(11, 16), sharex=True)
    fig.suptitle(f'Aquaponics + Vermiponics Simulation\nT = {T}°C | Initial Alk = {initial_Alk} mg/L CaCO₃')
    
    axs[0].plot(t, F, label='Fish (kg)'); axs[0].plot(t, P, label='Plants (kg)')
    axs[0].legend(); axs[0].set_ylabel('Biomass')
    
    axs[1].plot(t, A, label='NH₃-N'); axs[1].plot(t, I, label='NO₂-N')
    axs[1].plot(t, N, label='NO₃-N'); axs[1].plot(t, S, label='Org. solids')
    axs[1].legend(); axs[1].set_ylabel('mg/L')
    
    axs[2].plot(t, DO, 'tab:cyan', label='DO (mg/L)')
    axs[2].axhline(4.0, color='red', ls='--', label='Critical')
    axs[2].legend()
    
    axs[3].plot(t, Alk, label='Alkalinity (mg/L CaCO₃)'); axs[3].legend()
    
    axs[4].plot(t, pH_approx, 'purple', label='Approx pH'); axs[4].legend()
    axs[4].set_xlabel('Days')
    
    plt.tight_layout()
    plt.show()
    
    print(f"Final values (day 120):")
    print(f"  Fish: {F[-1]:.2f} kg")
    print(f"  Plants: {P[-1]:.2f} kg")
    print(f"  Nitrate: {N[-1]:.1f} mg/L")
    print(f"  DO: {DO[-1]:.1f} mg/L")
    print(f"  Organic solids: {S[-1]:.1f} mg/L")
    print(f"  Alkalinity: {Alk[-1]:.1f} mg/L")
    print(f"  Approx pH: {pH_approx[-1]:.1f}")

**Notes**
- Use the sliders above to explore how temperature and starting alkalinity affect system behavior.
- Low alkalinity leads to pH drop → reduced nitrification → potential ammonia buildup.
- Higher temperature accelerates metabolism/growth but increases oxygen demand.